# 🧪 CUB-200-2011 (LR=0.02, 20 EPOCHS): SO-KAC-RIDGE (FULL 31 ABLATIONS TRÊN VAL 80/20 & FINAL TEST TRÊN 100% TRAIN)
* **Ngày cập nhật:** 2026-10-21
* **Notebook gốc (Base):** `notebooks/apt/Cub200/M0_M1_M3.ipynb` & `notebooks/apt/Unified_SO_KAC_3ReLU_4Datasets.ipynb`
* **Mô hình Backbone:** APT (`ViT-B/16 Frozen` + First-Session Additive Prompt Tuning)
* **Mục tiêu nghiên cứu:** Thực thi quy trình **2 Giai đoạn chuẩn chống rò rỉ tập Test (Zero Test-Leakage)**:
  1. **Giai đoạn 1 (`phase="val"` — Chạy đủ 7 Nhóm / 31 Cấu hình Ablation trên Val 80/20):** Chia tập `train` gốc theo tỉ lệ **80% Train / 20% Val** qua **3 seed `0, 1, 2`** (Phương án A: mỗi seed vừa đảo thứ tự các lớp vào 10 Tasks, vừa lấy `Fold[seed]` trong 5-Fold Disjoint làm tập Val để 3 seed không trùng ảnh Val nào, phủ 60% dữ liệu Train). Tuyệt đối **không chạm vào tập Test**.
  2. **Giai đoạn 2 (`phase="test"` — Final Test CHỈ trên Model Chốt SO-KAC & Baselines Paper):** Sau khi hoàn tất Ablation và chốt mô hình trên Val, **train lại từ đầu trên 100% tập `train` gốc (gộp cả 80% Train + 20% Val)** và **đánh giá trên tập `test` chính thức** theo **3 seed `0, 1, 2`**. Final Test **CHỈ CHẠY trên Model chốt SO-KAC 3-ReLU và các Baselines đối chứng (`Linear Ridge`, `KAC Axis`, `RanPAC 10k/Iso-RAM/Iso-Dim`)**, tuyệt đối **KHÔNG chạy test trên các biến thể ablation** để bảo đảm tính khách quan và tiết kiệm thời gian GPU.

---
### ⚠️ BẢNG GHI NHẬN THAY ĐỔI & ĐỒNG NHẤT SIÊU THAM SỐ THEO PAPER APT (ICCV 2025)
1. **Kiến trúc Backbone & LayerNorm (Giữ nguyên 100% chuẩn gốc):**
   - `VIT.eval()` luôn cố định; `CosineHead` ở Phase 1 dùng `nn.LayerNorm(768, elementwise_affine=False)`; `AxisKACRidgeHead` dùng `nn.LayerNorm(768, elementwise_affine=True)`, `grid_range=(-2.0, 2.0)`, `num_grids=4`.
2. **Đồng nhất 100% Hyperparameters Chuẩn Repo Tác Giả APT (`experiments/*.sh`):**
   | Dataset | Số Lớp / Chia Task | Phase 1 Epochs | Prompt LR | EMA Coeff | Batch Size | Đường dẫn Kaggle Chuẩn |
   | :--- | :---: | :---: | :---: | :---: | :---: | :--- |
   | **CUB-200** | 200C ($10\text{T} \times 20\text{C}$) | `20` | **`0.020`** | `0.7` | `128` | `/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder` |
   | **CIFAR-100** | 100C ($10\text{T} \times 10\text{C}$) | `20` | **`0.004`** | `0.7` | `128` | `/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python` |
   | **Cars-196** | 196C ($9\text{T}\times 20\text{C} + 1\text{T}\times 16\text{C}$) | `50` | **`0.005`** | `0.7` | `128` | `/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder` |
   | **ImageNet-R** | 200C ($10\text{T} \times 20\text{C}$) | `30` | **`0.003`** | `0.8` | `128` | `/kaggle/input/datasets/thohintrnhngc/imagenet-r` |
3. **7 Nhóm Ablation Toàn Diện (31 Cấu Hình Độc Lập):**
   - **Group 0 (Baselines):** `B0_linear_ridge`, `B1_kac_axis`, `B4_ranpac_dim2304`, `B3_ranpac_ram5640`, `B2_ranpac_10k`, `SO_KAC_3ReLU`
   - **Group 1 (Linear Skip `[f, psi]`):** `A1a_no_skip_pure_nl`, `A1b_add_skip_768`, `A1c_add_tiled_1536`, `A1d_concat_no_center`, `SO_KAC_3ReLU`
   - **Group 2 (Orthogonal Dim):** `A2a_orth_384`, `A2b_orth_768`, `SO_KAC_3ReLU` (1536), `A2d_orth_2304`, `A2e_orth_3072`
   - **Group 3 (Number of Heads $H$):** `A3a_heads_1`, `A3b_heads_2`, `A3c_heads_4`, `SO_KAC_3ReLU` (6H), `A3e_heads_8`, `A3f_monolithic_9984`
   - **Group 4 (Spline Knots & Basis):** `A4a_1relu_only`, `A4b_2relu_knots`, `SO_KAC_3ReLU` (3-ReLU), `A4d_4relu_knots`, `A4e_4knots_tanh`, `A4f_full_v20`, `A4g_gaussian_rbf`
   - **Group 5 (Projection Matrix):** `A5a_proj_gaussian`, `A5b_proj_rademacher`, `SO_KAC_3ReLU` (Haar QR)
   - **Group 6 (Continual Alignment):** `A6a_no_align`, `A6b_only_rms`, `A6c_rms_plus_cb`, `A6d_rms_plus_wa`, `SO_KAC_3ReLU` (Full Align)
4. **Đo đạc Đầy Đủ 4 Chỉ Số Chẩn Đoán & Hiệu Năng Phần Cứng:**
   - `eff_rank` (ViT 768d), `eff_rank_phi` (Gram $\Phi$), `u_out_of_grid`, `acc_ta`, `p_grad`, `c_grad`, `gram_ram_mb`, `peak_vram_mb`, `total_gflops_10t`, `solve_time_s`, `infer_time_ms`.


In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG, THƯ VIỆN & THIẾT BỊ GPU
# ==============================================================================
import os
import sys
import math
import time
import json
import random
import pickle
import shutil
import subprocess
import warnings
from collections import OrderedDict
warnings.filterwarnings("ignore")

try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms as T
from torchvision.datasets import ImageFolder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"✅ PyTorch: {torch.__version__} | timm: {timm.__version__} | Device: {DEVICE}" +
      (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))


In [ ]:
# ==============================================================================
# CELL 3: CẤU HÌNH THỰC NGHIỆM (CFG) & DANH MỤC 7 NHÓM / 31 CẤU HÌNH ABLATION
# ==============================================================================
IS_KAGGLE = os.path.exists("/kaggle")

# 1. Chọn Dataset & Chế độ chạy:
DATASETS_TO_RUN  = ["cub200"]   # Có thể đặt ["cub200"], ["cars196"], ["cifar100"], ["imagenet_r"] hoặc cả 4
SEEDS_TO_RUN     = [0, 1, 2]       # Chuẩn 3 seeds [0, 1, 2]
RUN_VAL_ABLATION = True            # Giai đoạn 1: Chạy đủ 31 Ablations trên tập Val 20% (học trên 80% Train, KHÔNG chạm Test)
RUN_FINAL_TEST   = True            # Giai đoạn 2: Chạy Final Benchmark trên tập Test chính thức (học lại trên 100% Train gốc)
MAX_TASKS_DEBUG  = None            # None = chạy đủ 10 tasks; đặt 2 nếu muốn kiểm tra nhanh luồng code

# 2. Thông số chuẩn 100% theo Repo APT (ICCV 2025) cho từng Dataset:
DATASET_SPECS = OrderedDict([
    ("cub200", dict(
        dataset_name="Cub200", n_classes=200, task_sizes=[20]*10,
        first_session_epochs=20, prompt_lr=0.020, ema_coeff=0.7, batch_size=128, aug_type="imagenet",
        kaggle_candidates=[
            "/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
            "/kaggle/input/cub-200-2011-by-classes-folder",
            "/kaggle/input/cyizhuo-cub-200-2011-by-classes-folder",
            "./data/cub200",
        ]
    )),
    ("cifar100", dict(
        dataset_name="Cifar100", n_classes=100, task_sizes=[10]*10,
        first_session_epochs=20, prompt_lr=0.004, ema_coeff=0.7, batch_size=128, aug_type="cifar10",
        kaggle_candidates=[
            "/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
            "/kaggle/input/cifar-100-python/cifar-100-python",
            "/kaggle/input/cifar-100-python",
            "./data/cifar-100-python",
        ]
    )),
    ("cars196", dict(
        dataset_name="Cars196", n_classes=196, task_sizes=[20]*9 + [16],
        first_session_epochs=50, prompt_lr=0.005, ema_coeff=0.7, batch_size=128, aug_type="imagenet",
        kaggle_candidates=[
            "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
            "/kaggle/input/stanford-car-dataset-by-classes-folder/car_data/car_data",
            "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data",
            "./data/car_data/car_data",
        ]
    )),
    ("imagenet_r", dict(
        dataset_name="ImageNet_R", n_classes=200, task_sizes=[20]*10,
        first_session_epochs=30, prompt_lr=0.003, ema_coeff=0.8, batch_size=128, aug_type="imnet_r",
        kaggle_candidates=[
            "/kaggle/input/datasets/thohintrnhngc/imagenet-r",
            "/kaggle/input/datasets/thohintrnhngc/imagenet-r/imagenet-r",
            "/kaggle/input/imagenet-r",
            "/kaggle/input/imagenet-r/imagenet-r",
            "./data/imagenet-r",
        ]
    )),
])

RIDGE_LAMBDAS = [v for p in range(-8, 9) for v in (10.0**p, 3.0 * (10.0**p))]
TIMM_NAME     = "vit_base_patch16_224.augreg_in21k_ft_in1k"
USE_AMP       = True
NUM_WORKERS   = 2 if IS_KAGGLE else 0
MASTER_OUT_DIR = "/kaggle/working/outputs/so_kac_master" if IS_KAGGLE else "./results/apt/so_kac_master"
os.makedirs(MASTER_OUT_DIR, exist_ok=True)

# 3. Danh mục 31 Cấu hình Ablation (Tâm điểm: SO_KAC_3ReLU chuẩn 3-ReLU B-Spline):
ABLATION_REGISTRY = OrderedDict([
    # --- Nhóm 0: Baselines ---
    ("B0_linear_ridge",     dict(arch="linear", M=768, desc="Linear Ridge gốc trên đặc trưng ViT 768 chiều (1x768)")),
    ("B1_kac_axis",         dict(arch="kac_axis", num_grids=4, grid_range=(-2.0, 2.0), desc="KAC 1D dọc 768 trục gốc (1x3840, không xoay Haar)")),
    ("B4_ranpac_dim2304",   dict(arch="ranpac", M=2304, desc="RanPAC cùng chiều 1 Head SO-KAC (1x2304 Gaussian RP)")),
    ("B3_ranpac_ram5640",   dict(arch="ranpac", M=5640, desc="RanPAC cùng ngân sách 121.3 MB RAM với SO-KAC (1x5640)")),
    ("B2_ranpac_10k",       dict(arch="ranpac", M=10000, desc="RanPAC Chuẩn NeurIPS 2023 (1x10000 Gaussian RP, 381.5 MB RAM)")),

    # --- Cấu hình Đề xuất Chuẩn (Anchor cho toàn bộ 7 Nhóm) ---
    ("SO_KAC_3ReLU",        dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="★ SO-KAC Đề Xuất Chuẩn (6H x [768+1536], Haar QR, 3-ReLU B-Spline, Full Align)")),

    # --- Nhóm 1: Vai trò của Nhánh Nối Tắt 768 chiều gốc [f, psi] & Kernel Centering ---
    ("A1a_no_skip_pure_nl", dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="none", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Skip] KHÔNG nối 768 chiều gốc (Chỉ dùng nhánh phi tuyến 6H x 1536)")),
    ("A1b_add_skip_768",    dict(arch="so_kac", num_heads=6, orth_dim=768, skip_mode="add_768", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Skip] CỘNG ĐÈ f + psi_orth trên 768 chiều (6H x 768, giao thoa hệ trục)")),
    ("A1c_add_tiled_1536",  dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="add_1536", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Skip] CỘNG ĐÈ [f,f]/sqrt(2) + psi_orth trên 1536 chiều (6H x 1536)")),
    ("A1d_concat_no_center",dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=False,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Skip] Ghép nối [f, psi] nhưng TẮT Kernel Centering (Nhiễu DC-bias dương)")),

    # --- Nhóm 2: Số chiều trực giao Haar mỗi khối (orth_dim = 384 -> 3072) ---
    ("A2a_orth_384",        dict(arch="so_kac", num_heads=6, orth_dim=384, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[OrthDim] orth_dim = 384 -> D_h = 1,152 (30.4 MB RAM)")),
    ("A2b_orth_768",        dict(arch="so_kac", num_heads=6, orth_dim=768, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[OrthDim] orth_dim = 768 -> D_h = 1,536 (54.0 MB RAM)")),
    ("A2d_orth_2304",       dict(arch="so_kac", num_heads=6, orth_dim=2304, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[OrthDim] orth_dim = 2304 -> D_h = 3,072 (216.0 MB RAM)")),
    ("A2e_orth_3072",       dict(arch="so_kac", num_heads=6, orth_dim=3072, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[OrthDim] orth_dim = 3072 -> D_h = 3,840 (337.5 MB RAM)")),

    # --- Nhóm 3: Số lượng Heads (H = 1..8) vs 1 Khối Đặc Monolithic (9,984 chiều) ---
    ("A3a_heads_1",         dict(arch="so_kac", num_heads=1, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Heads] H = 1 Head x 2,304 (20.2 MB RAM)")),
    ("A3b_heads_2",         dict(arch="so_kac", num_heads=2, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Heads] H = 2 Heads x 2,304 (40.5 MB RAM)")),
    ("A3c_heads_4",         dict(arch="so_kac", num_heads=4, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Heads] H = 4 Heads x 2,304 (81.0 MB RAM)")),
    ("A3e_heads_8",         dict(arch="so_kac", num_heads=8, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Heads] H = 8 Heads x 2,304 (162.0 MB RAM)")),
    ("A3f_monolithic_9984", dict(arch="so_kac", num_heads=1, orth_dim=9216, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Heads] H = 1 Khối Đặc Monolithic (D_h = 9,984 = 768+9216 | 380.2 MB RAM)")),

    # --- Nhóm 4: Hàm cơ sở phi tuyến (1-ReLU -> 2-ReLU -> 3-ReLU -> 4-ReLU -> +Tanh -> +Hadamard vs Gaussian RBF) ---
    ("A4a_1relu_only",      dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="1_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Basis] Chỉ 1 hàm ReLU duy nhất: s0 = ReLU(z)")),
    ("A4b_2relu_knots",     dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="2_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Basis] 2 hàm ReLU Knots: s0 + alpha*(0.45*s1)")),
    ("A4d_4relu_knots",     dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="4_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Basis] 4 hàm Monotonic B-Spline Knots: s0, s1, s2, s3")),
    ("A4e_4knots_tanh",     dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="4_relu_tanh", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Basis] 4 Knots B-Spline + Integrated RBF tanh(s0)")),
    ("A4f_full_v20",        dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="full_v20", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Basis] 4 Knots + tanh(s0) + tích chéo Hadamard (Bản V20 cũ)")),
    ("A4g_gaussian_rbf",    dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="gaussian_rbf", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Basis] Thay B-Spline bằng 4 hàm chuông Gaussian RBF cổ điển (Sụp đuôi)")),

    # --- Nhóm 5: Bản chất Ma trận chiếu (Gaussian i.i.d. vs Rademacher vs Haar Orthogonal QR) ---
    ("A5a_proj_gaussian",   dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="gaussian_iid", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Proj] Chiếu Ngẫu nhiên Gaussian i.i.d. N(0, 1/D)")),
    ("A5b_proj_rademacher", dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="rademacher", use_rms=True, cb_gamma=0.32, wa_beta=0.80,
                                 desc="[Proj] Chiếu Nhị phân Rademacher +-1/sqrt(D)")),

    # --- Nhóm 6: Cơ chế Cân bằng liên Task (Global RMS, Class-Prior Balancing & Task-Energy WA) ---
    ("A6a_no_align",        dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=False, cb_gamma=0.0, wa_beta=0.0,
                                 desc="[Align] Tắt cả 3: Không Global RMS, Không Class-Balance, Không Task-WA")),
    ("A6b_only_rms",        dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.0, wa_beta=0.0,
                                 desc="[Align] Chỉ bật Global RMS Calibration")),
    ("A6c_rms_plus_cb",     dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.32, wa_beta=0.0,
                                 desc="[Align] Bật Global RMS + Class-Prior Balancing (gamma=0.32, tắt Task-WA)")),
    ("A6d_rms_plus_wa",     dict(arch="so_kac", num_heads=6, orth_dim=1536, skip_mode="concat", use_centering=True,
                                 basis_mode="3_relu", proj_mode="haar_qr", use_rms=True, cb_gamma=0.0, wa_beta=0.80,
                                 desc="[Align] Bật Global RMS + Task-Energy WA (beta=0.80, tắt Class-Balance)")),
])

ABLATION_GROUPS = OrderedDict([
    ("group_0_baselines",       ("NHÓM 0: ĐỐI CHIẾU BASELINES & RANPAC CÙNG NGÂN SÁCH BỘ NHỚ",
                                 ["B0_linear_ridge", "B1_kac_axis", "B4_ranpac_dim2304", "B3_ranpac_ram5640", "B2_ranpac_10k", "SO_KAC_3ReLU"])),
    ("group_1_linear_skip",     ("NHÓM 1: VAI TRÒ CỦA NHÁNH NỐI TẮT 768 CHIỀU GỐC [f, psi] & KERNEL CENTERING",
                                 ["A1a_no_skip_pure_nl", "A1b_add_skip_768", "A1c_add_tiled_1536", "A1d_concat_no_center", "SO_KAC_3ReLU", "B2_ranpac_10k"])),
    ("group_2_orth_dim",        ("NHÓM 2: KHẢO SÁT SỐ CHIỀU CHIẾU TRỰC GIAO HAAR MỖI KHỐI (orth_dim = 384 -> 3072)",
                                 ["A2a_orth_384", "A2b_orth_768", "SO_KAC_3ReLU", "A2d_orth_2304", "A2e_orth_3072", "B2_ranpac_10k"])),
    ("group_3_num_heads",       ("NHÓM 3: KHẢO SÁT SỐ KHỐI ĐƯỜNG CHÉO (H = 1..8) VS 1 KHỐI ĐẶC MONOLITHIC (9,984 CHIỀU)",
                                 ["A3a_heads_1", "A3b_heads_2", "A3c_heads_4", "SO_KAC_3ReLU", "A3e_heads_8", "A3f_monolithic_9984", "B2_ranpac_10k"])),
    ("group_4_basis_functions", ("NHÓM 4: KHẢO SÁT HÀM RELU KNOTS (1..4), TANH, HADAMARD & GAUSSIAN RBF",
                                 ["A4a_1relu_only", "A4b_2relu_knots", "SO_KAC_3ReLU", "A4d_4relu_knots", "A4e_4knots_tanh", "A4f_full_v20", "A4g_gaussian_rbf", "B2_ranpac_10k"])),
    ("group_5_projection_type", ("NHÓM 5: BẢN CHẤT MA TRẬN CHIẾU (GAUSSIAN I.I.D. VS RADEMACHER VS HAAR ORTHOGONAL QR)",
                                 ["A5a_proj_gaussian", "A5b_proj_rademacher", "SO_KAC_3ReLU", "B2_ranpac_10k"])),
    ("group_6_alignment",       ("NHÓM 6: BÓC TÁCH ĐÓNG GÓP CỦA GLOBAL RMS, CLASS-PRIOR BALANCING & TASK WEIGHT ALIGNING",
                                 ["A6a_no_align", "A6b_only_rms", "A6c_rms_plus_cb", "A6d_rms_plus_wa", "SO_KAC_3ReLU", "B2_ranpac_10k"])),
])

ALL_ABLATION_IDS  = list(ABLATION_REGISTRY.keys())
# Final Test CHỈ chạy trên Model chốt SO-KAC 3-ReLU và các Baselines Replicate Paper (KHÔNG chạy trên các biến thể Ablation):
FINAL_TEST_MODELS = [
    "B0_linear_ridge",     # Linear Analytic Ridge Baseline
    "B1_kac_axis",         # KAC Splines (Axis) Baseline
    "B4_ranpac_dim2304",   # RanPAC Iso-Dim (2,304 dims)
    "B3_ranpac_ram5640",   # RanPAC Iso-RAM (5,640 dims, khớp 121.5 MB RAM của SO-KAC)
    "B2_ranpac_10k",       # RanPAC Chuẩn Paper NeurIPS 2023 (10,000 dims)
    "SO_KAC_3ReLU",        # ★ MODEL CHỐT SO-KAC ĐỀ XUẤT (6H x 2,304 dims, Haar QR, 3-ReLU, Full Align)
]
print(f"✅ Đã đăng ký {len(ALL_ABLATION_IDS)} cấu hình Ablation (7 Nhóm cho Giai đoạn Val) | {len(FINAL_TEST_MODELS)} mô hình Final Test chính thức.")
print(f"📌 Datasets cần chạy: {DATASETS_TO_RUN} | Seeds: {SEEDS_TO_RUN} | Val Ablation: {RUN_VAL_ABLATION} | Final Test: {RUN_FINAL_TEST}")


In [ ]:
# ==============================================================================
# CELL 4: NẠP DỮ LIỆU TRỰC TIẾP TỪ KAGGLE & CHIA DISJOINT TRAIN/VAL (PHƯƠNG ÁN A)
# ==============================================================================
class CIFAR100PickleDataset(Dataset):
    """Đọc trực tiếp 3 file pickle train/test/meta của CIFAR-100 có sẵn trên Kaggle."""
    def __init__(self, root, train=True):
        split_file = "train" if train else "test"
        with open(os.path.join(root, split_file), "rb") as f:
            entry = pickle.load(f, encoding="latin1")
        with open(os.path.join(root, "meta"), "rb") as f:
            meta = pickle.load(f, encoding="latin1")
        self.data = entry["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # (N, 32, 32, 3) uint8
        self.targets = [int(y) for y in entry["fine_labels"]]
        self.classes = list(meta["fine_label_names"])

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return Image.fromarray(self.data[idx]), self.targets[idx]

class RemappedSubset(Dataset):
    """Bọc Dataset gốc theo danh sách chỉ số `indices`, ánh xạ lại nhãn sang [0..C-1] theo `remap` và áp dụng `tf`."""
    def __init__(self, base_dataset, indices, remap, tf):
        self.base = base_dataset
        self.indices = list(indices)
        self.remap = remap
        self.tf = tf
        base_targets = np.asarray(base_dataset.targets)
        self.targets = [int(remap[int(base_targets[i])]) for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        raw_idx = self.indices[i]
        if isinstance(self.base, CIFAR100PickleDataset):
            img = Image.fromarray(self.base.data[raw_idx])
        else:
            path, _ = self.base.samples[raw_idx]
            with Image.open(path) as im:
                img = im.convert("RGB")
        return self.tf(img), self.targets[i]

def build_transforms(aug_type):
    if aug_type == "cifar10":
        train_tf = T.Compose([
            T.AutoAugment(T.AutoAugmentPolicy.CIFAR10, T.InterpolationMode.BILINEAR),
            T.RandomResizedCrop(224),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
        ])
    elif aug_type == "imnet_r":
        train_tf = T.Compose([
            T.Resize(256),
            T.RandomResizedCrop(224),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
        ])
    else:  # "imagenet" (CUB-200 & Stanford Cars-196)
        train_tf = T.Compose([
            T.AutoAugment(T.AutoAugmentPolicy.IMAGENET, T.InterpolationMode.BILINEAR),
            T.RandomResizedCrop(224),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
        ])
    test_tf = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
    ])
    return train_tf, test_tf

def load_raw_dataset(d_key):
    spec = DATASET_SPECS[d_key]
    if d_key == "cifar100":
        root = next((p for p in spec["kaggle_candidates"] if os.path.isdir(p) and os.path.isfile(os.path.join(p, "train"))), None)
        if root is None:
            return None, None, None
        return CIFAR100PickleDataset(root, train=True), CIFAR100PickleDataset(root, train=False), root
    else:
        root = next((p for p in spec["kaggle_candidates"] if os.path.isdir(os.path.join(p, "train")) and os.path.isdir(os.path.join(p, "test"))), None)
        if root is None:
            return None, None, None
        return ImageFolder(os.path.join(root, "train")), ImageFolder(os.path.join(root, "test")), root

def setup_cil_seed(train_dataset, seed, num_classes, task_sizes, n_splits=5, base_seed=1993):
    """
    Phương án A (Cách 2):
      1. Đảo thứ tự các lớp vào 10 Tasks theo `seed` (np.random.RandomState(seed)).
      2. Chia Stratified Disjoint 5-Fold trên `train_dataset` (base_seed=1993):
         - Lấy `folds[seed]` làm Val (20%), 4 fold còn lại làm Train (80%).
         - Đảm bảo 3 tập Val của seed 0, 1, 2 hoàn toàn KHÔNG TRÙNG NHAU (Overlap = 0%).
    """
    assert 0 <= seed < n_splits, f"seed phải nằm trong [0, {n_splits-1}]"
    rng_task = np.random.RandomState(seed)
    class_order = rng_task.permutation(num_classes).tolist()
    remap = {orig_c: new_c for new_c, orig_c in enumerate(class_order)}

    bounds, cur = [], 0
    for sz in task_sizes:
        bounds.append((cur, cur + sz))
        cur += sz

    targets = np.asarray(train_dataset.targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        c_indices = np.where(targets == c)[0]
        perm = rng_split.permutation(c_indices)
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed:
                tr_idx.extend(folds[k])

    return class_order, remap, bounds, np.array(tr_idx), np.array(va_idx)

# Kiểm tra các dataset khả dụng và xác thực 0% Overlap giữa 3 seeds:
AVAILABLE_DATASETS = OrderedDict()
for d_key in DATASETS_TO_RUN:
    tr_ds, te_ds, root_p = load_raw_dataset(d_key)
    spec = DATASET_SPECS[d_key]
    if tr_ds is None:
        print(f"⚠️ [BỎ QUA] {spec['dataset_name']}: Không tìm thấy trong {spec['kaggle_candidates']}")
        continue
    AVAILABLE_DATASETS[d_key] = (tr_ds, te_ds, root_p)
    val_sets = {}
    for s in [0, 1, 2]:
        _, _, _, tr_i, va_i = setup_cil_seed(tr_ds, s, spec["n_classes"], spec["task_sizes"])
        val_sets[s] = set(va_i.tolist())
    ov = len(val_sets[0] & val_sets[1]) + len(val_sets[1] & val_sets[2]) + len(val_sets[0] & val_sets[2])
    unq = len(val_sets[0] | val_sets[1] | val_sets[2])
    print(f"✅ [{spec['dataset_name']:<10}] Root: {root_p}")
    print(f"   Train gốc (100%): {len(tr_ds):>6} | Test (Khóa): {len(te_ds):>6} | "
          f"Improve 80/20: Train={len(tr_i)}, Val={len(va_i)} | Overlap Val 3 seeds={ov} | Phủ Val={unq}/{len(tr_ds)} ({100*unq/len(tr_ds):.1f}%)")


In [ ]:
# ==============================================================================
# CELL 5: KIẾN TRÚC BACKBONE VIT-B/16 (FROZEN), APT PROMPT & HUẤN LUYỆN TASK 1
# ==============================================================================
class PatchEmbed(nn.Module):
    def __init__(self, dim=768, patch=16):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch, patch)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class Attention(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__()
        self.heads, self.hd = heads, dim // heads
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x, prompt=None):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.heads, self.hd).permute(2, 0, 3, 1, 4).unbind(0)
        if prompt is not None:
            pk, pv = prompt
            k = torch.cat([k[:, :, :1] + pk.to(k.dtype), k[:, :, 1:]], dim=2)
            v = torch.cat([v[:, :, :1] + pv.to(v.dtype), v[:, :, 1:]], dim=2)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(B, N, C))

class Mlp(nn.Module):
    def __init__(self, dim=768, hidden=3072):
        super().__init__()
        self.fc1, self.act, self.fc2 = nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, dim * 4)
    def forward(self, x, prompt=None):
        x = x + self.attn(self.norm1(x), prompt)
        return x + self.mlp(self.norm2(x))

class ViT(nn.Module):
    def __init__(self, dim=768, depth=12, heads=12):
        super().__init__()
        self.patch_embed = PatchEmbed(dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 197, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim, eps=1e-6)
    def forward(self, x, prompt_fn=None):
        B = x.size(0)
        x = torch.cat([self.cls_token.expand(B, -1, -1), self.patch_embed(x)], dim=1) + self.pos_embed
        for l, blk in enumerate(self.blocks):
            x = blk(x, prompt_fn(l) if prompt_fn else None)
        return self.norm(x)[:, 0]

def load_pretrained_vit(name):
    try:
        ref = timm.create_model(name, pretrained=True, num_classes=0)
    except Exception:
        ref = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
    sd = ref.state_dict()
    vit = ViT(768, 12, 12)
    with torch.no_grad():
        vit.patch_embed.proj.weight.copy_(sd["patch_embed.proj.weight"])
        vit.patch_embed.proj.bias.copy_(sd["patch_embed.proj.bias"])
        vit.cls_token.copy_(sd["cls_token"])
        vit.pos_embed.copy_(sd["pos_embed"])
        vit.norm.weight.copy_(sd["norm.weight"])
        vit.norm.bias.copy_(sd["norm.bias"])
        for i in range(12):
            vit.blocks[i].norm1.weight.copy_(sd[f"blocks.{i}.norm1.weight"])
            vit.blocks[i].norm1.bias.copy_(sd[f"blocks.{i}.norm1.bias"])
            vit.blocks[i].attn.qkv.weight.copy_(sd[f"blocks.{i}.attn.qkv.weight"])
            vit.blocks[i].attn.qkv.bias.copy_(sd[f"blocks.{i}.attn.qkv.bias"])
            vit.blocks[i].attn.proj.weight.copy_(sd[f"blocks.{i}.attn.proj.weight"])
            vit.blocks[i].attn.proj.bias.copy_(sd[f"blocks.{i}.attn.proj.bias"])
            vit.blocks[i].norm2.weight.copy_(sd[f"blocks.{i}.norm2.weight"])
            vit.blocks[i].norm2.bias.copy_(sd[f"blocks.{i}.norm2.bias"])
            vit.blocks[i].mlp.fc1.weight.copy_(sd[f"blocks.{i}.mlp.fc1.weight"])
            vit.blocks[i].mlp.fc1.bias.copy_(sd[f"blocks.{i}.mlp.fc1.bias"])
            vit.blocks[i].mlp.fc2.weight.copy_(sd[f"blocks.{i}.mlp.fc2.weight"])
            vit.blocks[i].mlp.fc2.bias.copy_(sd[f"blocks.{i}.mlp.fc2.bias"])
    for p in vit.parameters():
        p.requires_grad = False
    return vit.eval()

VIT = load_pretrained_vit(TIMM_NAME).to(DEVICE)
print("✅ ViT-B/16 Backbone đã nạp và đóng băng 100% (eval mode).")

class APTPrompt(nn.Module):
    def __init__(self, depth=12, dim=768, heads=12, ema_coeff=0.7):
        super().__init__()
        self.depth, self.heads, self.hd = depth, heads, dim // heads
        self.ema_coeff = ema_coeff
        self.prompt_tokens = nn.Parameter(torch.zeros(depth * 2, dim))
        nn.init.trunc_normal_(self.prompt_tokens, std=0.02)
    def get(self, l):
        P = self.prompt_tokens
        return P[2 * l].view(self.heads, 1, self.hd), P[2 * l + 1].view(self.heads, 1, self.hd)

class CosineHead(nn.Module):
    def __init__(self, dim=768, num_classes=20):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.fc = nn.Linear(dim, num_classes, bias=False)
    def forward(self, f):
        return self.norm(f) @ F.normalize(self.fc.weight, dim=1).t()

def train_first_session_prompt(task1_dataset, n_cls_t1, epochs, lr, ema, batch_size):
    """Huấn luyện APT Prompt ở Task 1 và trả về (prompt_frozen, avg_p_grad, avg_c_grad)."""
    prompt = APTPrompt(12, 768, 12, ema_coeff=ema).to(DEVICE)
    temp_head = CosineHead(dim=768, num_classes=n_cls_t1).to(DEVICE)
    opt = torch.optim.Adam([
        {"params": [prompt.prompt_tokens], "lr": lr},
        {"params": list(temp_head.parameters()), "lr": lr}
    ], betas=(0.9, 0.999))

    bs1 = min(batch_size, max(len(task1_dataset), 1))
    loader1 = DataLoader(task1_dataset, batch_size=bs1, shuffle=True, drop_last=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and DEVICE == "cuda")

    p_grads, c_grads = [], []
    for ep in range(epochs):
        VIT.eval(); temp_head.train(); prompt.train()
        tot_loss, ep_pg, ep_cg, n_b = 0.0, 0.0, 0.0, 0
        for x, y in loader1:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE == "cuda"):
                feat = VIT(x, prompt_fn=prompt.get)
                logits = temp_head(feat)
                loss = F.cross_entropy(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            if prompt.prompt_tokens.grad is not None:
                ep_pg += prompt.prompt_tokens.grad.detach().norm(2).item()
            if temp_head.fc.weight.grad is not None:
                ep_cg += temp_head.fc.weight.grad.detach().norm(2).item()
            n_b += 1
            scaler.update()
            tot_loss += loss.item()
        p_grads.append(ep_pg / max(n_b, 1))
        c_grads.append(ep_cg / max(n_b, 1))
        if (ep + 1) % 10 == 0 or (ep + 1) == epochs:
            print(f"    [Task 1 Prompt] Ep {ep+1:2d}/{epochs} | Loss: {tot_loss/max(n_b, 1):.4f} | p_grad: {p_grads[-1]:.4f} | c_grad: {c_grads[-1]:.4f}")

    for p in prompt.parameters():
        p.requires_grad = False
    prompt.eval()
    del temp_head
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return prompt, round(float(np.mean(p_grads)), 4), round(float(np.mean(c_grads)), 4)


In [ ]:
# ==============================================================================
# CELL 6: ĐỊNH NGHĨA CÁC BỘ PHÂN LOẠI NGHIỆM ĐÓNG & CONFIGURABLE SO-KAC HEAD
# ==============================================================================
class BaseRidgeHead(nn.Module):
    def __init__(self, M_unique, num_heads, head_dim, num_classes=200):
        super().__init__()
        self.M = M_unique
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.num_classes = num_classes
        self.optimal_lambda = 1000.0
        self.total_flops_accum = 0.0

    def find_optimal_lambda_fast(self, task_h_2d, task_y, candidate_lambdas, min_lambda=100.0):
        g = torch.Generator().manual_seed(1993)
        y_cpu = task_y.detach().cpu()
        tr_idx, val_idx = [], []
        for c in torch.unique(y_cpu).tolist():
            idx_c = (y_cpu == c).nonzero(as_tuple=True)[0]
            if len(idx_c) == 0:
                continue
            idx_c = idx_c[torch.randperm(len(idx_c), generator=g)]
            n_tr_c = max(int(len(idx_c) * 0.8), 1)
            tr_idx.append(idx_c[:n_tr_c])
            if len(idx_c) > n_tr_c:
                val_idx.append(idx_c[n_tr_c:])
        if len(tr_idx) == 0:
            tr_idx = torch.arange(len(task_y), device=task_h_2d.device)
            val_idx = tr_idx
        else:
            tr_idx = torch.cat(tr_idx).to(task_h_2d.device)
            val_idx = torch.cat(val_idx).to(task_h_2d.device) if len(val_idx) > 0 else tr_idx

        h_tr = task_h_2d[tr_idx].to(torch.float64)
        y_tr = F.one_hot(task_y[tr_idx], num_classes=self.num_classes).to(torch.float64)
        h_val = task_h_2d[val_idx].to(torch.float64)
        y_val = F.one_hot(task_y[val_idx], num_classes=self.num_classes).to(torch.float64)
        self.cv_losses = {}

        N_tr, D_h = h_tr.shape
        best_loss = float("inf")
        best_ridge = min_lambda

        if N_tr <= D_h:
            K_tr = h_tr @ h_tr.T
            K_val_tr = h_val @ h_tr.T
            I_N = torch.eye(N_tr, device=task_h_2d.device, dtype=torch.float64)
            for r in candidate_lambdas:
                if r < min_lambda:
                    continue
                try:
                    L = torch.linalg.cholesky(K_tr + r * I_N)
                    alpha_r = torch.cholesky_solve(y_tr, L)
                    pred = K_val_tr @ alpha_r
                    loss = F.mse_loss(pred, y_val).item()
                    self.cv_losses[r] = loss
                    if loss < best_loss:
                        best_loss = loss
                        best_ridge = r
                except Exception:
                    continue
        else:
            G_tr = h_tr.T @ h_tr
            Q_tr = h_tr.T @ y_tr
            I_D = torch.eye(D_h, device=task_h_2d.device, dtype=torch.float64)
            for r in candidate_lambdas:
                if r < min_lambda:
                    continue
                try:
                    L = torch.linalg.cholesky(G_tr + r * I_D)
                    W_r = torch.cholesky_solve(Q_tr, L)
                    pred = h_val @ W_r
                    loss = F.mse_loss(pred, y_val).item()
                    self.cv_losses[r] = loss
                    if loss < best_loss:
                        best_loss = loss
                        best_ridge = r
                except Exception:
                    continue
        return best_ridge

class LinearRidgeHead(BaseRidgeHead):
    def __init__(self, in_features=768, num_classes=200):
        super().__init__(in_features, 1, in_features, num_classes)
        self.in_features = in_features
        self.register_buffer("G", torch.zeros(1, in_features, in_features, dtype=torch.float32))
        self.register_buffer("Q", torch.zeros(1, in_features, num_classes, dtype=torch.float32))
        self.register_buffer("weight", torch.zeros(num_classes, in_features, dtype=torch.float32))

    def expand_features(self, f):
        return f.float()

    @torch.no_grad()
    def update_and_solve(self, task_f, task_y, candidate_lambdas=None, task_range=None):
        h = self.expand_features(task_f)
        N = h.shape[0]
        Y = F.one_hot(task_y, num_classes=self.num_classes).to(torch.float32)
        self.optimal_lambda = self.find_optimal_lambda_fast(h, task_y, candidate_lambdas, min_lambda=0.0)
        self.G[0] += h.T @ h
        self.Q[0] += h.T @ Y
        I_D = torch.eye(self.head_dim, device=self.G.device, dtype=torch.float64)
        G_reg = self.G[0].to(torch.float64) + self.optimal_lambda * I_D
        Q_64 = self.Q[0].to(torch.float64)
        L, info = torch.linalg.cholesky_ex(G_reg)
        W_opt = torch.cholesky_solve(Q_64, L) if info.item() == 0 else torch.linalg.solve(G_reg, Q_64)
        self.weight.copy_(W_opt.T.float())
        self.total_flops_accum += (2.0 * N * (self.head_dim**2) + 2.0 * N * self.head_dim * self.num_classes +
                                   (1.0 / 3.0) * (self.head_dim**3) + 2.0 * (self.head_dim**2) * self.num_classes)

    def forward(self, f):
        return self.expand_features(f) @ self.weight.T

class AxisKACRidgeHead(BaseRidgeHead):
    def __init__(self, in_features=768, num_classes=200, num_grids=4, grid_range=(-2.0, 2.0)):
        M = in_features * (1 + num_grids)
        super().__init__(M, 1, M, num_classes)
        self.in_features = in_features
        self.num_grids = num_grids
        self.grid_range = grid_range
        self.layernorm = nn.LayerNorm(in_features, elementwise_affine=True)
        grid = torch.linspace(grid_range[0], grid_range[1], num_grids)
        self.register_buffer("grid", grid)
        self.denominator = (grid_range[1] - grid_range[0]) / max(num_grids - 1, 1)
        self.register_buffer("G", torch.zeros(1, M, M, dtype=torch.float32))
        self.register_buffer("Q", torch.zeros(1, M, num_classes, dtype=torch.float32))
        self.register_buffer("weight", torch.zeros(num_classes, M, dtype=torch.float32))

    def expand_features(self, f):
        u = self.layernorm(f.float())
        dist = (u[..., None] - self.grid) / self.denominator
        basis = torch.exp(-dist.pow(2)).flatten(1)
        return torch.cat([u, basis], dim=-1)

    @torch.no_grad()
    def update_and_solve(self, task_f, task_y, candidate_lambdas=None, task_range=None):
        h = self.expand_features(task_f)
        N = h.shape[0]
        Y = F.one_hot(task_y, num_classes=self.num_classes).to(torch.float32)
        self.optimal_lambda = self.find_optimal_lambda_fast(h, task_y, candidate_lambdas, min_lambda=0.0)
        self.G[0] += h.T @ h
        self.Q[0] += h.T @ Y
        I_D = torch.eye(self.head_dim, device=self.G.device, dtype=torch.float64)
        G_reg = self.G[0].to(torch.float64) + self.optimal_lambda * I_D
        Q_64 = self.Q[0].to(torch.float64)
        L, info = torch.linalg.cholesky_ex(G_reg)
        W_opt = torch.cholesky_solve(Q_64, L) if info.item() == 0 else torch.linalg.solve(G_reg, Q_64)
        self.weight.copy_(W_opt.T.float())
        self.total_flops_accum += (2.0 * N * (self.head_dim**2) + 2.0 * N * self.head_dim * self.num_classes +
                                   (1.0 / 3.0) * (self.head_dim**3) + 2.0 * (self.head_dim**2) * self.num_classes)

    def forward(self, f):
        return self.expand_features(f) @ self.weight.T

class RanPACHead(BaseRidgeHead):
    def __init__(self, in_features=768, num_classes=200, M=10000, seed=1993):
        super().__init__(M, 1, M, num_classes)
        self.in_features = in_features
        self.register_buffer("G", torch.zeros(1, M, M, dtype=torch.float32))
        self.register_buffer("Q", torch.zeros(1, M, num_classes, dtype=torch.float32))
        self.register_buffer("weight", torch.zeros(num_classes, M, dtype=torch.float32))
        g = torch.Generator().manual_seed(seed)
        self.register_buffer("W_rand", torch.randn(in_features, M, generator=g))

    def expand_features(self, f):
        return F.relu(f.float() @ self.W_rand)

    @torch.no_grad()
    def update_and_solve(self, task_f, task_y, candidate_lambdas=None, task_range=None):
        h = self.expand_features(task_f)
        N = h.shape[0]
        Y = F.one_hot(task_y, num_classes=self.num_classes).to(torch.float32)
        self.optimal_lambda = self.find_optimal_lambda_fast(h, task_y, candidate_lambdas, min_lambda=0.0)
        self.G[0] += h.T @ h
        self.Q[0] += h.T @ Y
        I_D = torch.eye(self.head_dim, device=self.G.device, dtype=torch.float64)
        G_reg = self.G[0].to(torch.float64) + self.optimal_lambda * I_D
        Q_64 = self.Q[0].to(torch.float64)
        L, info = torch.linalg.cholesky_ex(G_reg)
        W_opt = torch.cholesky_solve(Q_64, L) if info.item() == 0 else torch.linalg.solve(G_reg, Q_64)
        self.weight.copy_(W_opt.T.float())
        self.total_flops_accum += (2.0 * N * self.in_features * self.head_dim +
                                   2.0 * N * (self.head_dim**2) + 2.0 * N * self.head_dim * self.num_classes +
                                   (1.0 / 3.0) * (self.head_dim**3) + 2.0 * (self.head_dim**2) * self.num_classes)

    def forward(self, f):
        return self.expand_features(f) @ self.weight.T

class ConfigurableSOKACHead(BaseRidgeHead):
    def __init__(
        self,
        in_features=768,
        num_classes=200,
        num_heads=6,
        orth_dim=1536,
        skip_mode="concat",         # "concat", "none", "add_768", "add_1536"
        use_centering=True,         # True / False
        basis_mode="3_relu",        # "1_relu", "2_relu", "3_relu", "4_relu", "4_relu_tanh", "full_v20", "gaussian_rbf"
        proj_mode="haar_qr",        # "haar_qr", "gaussian_iid", "rademacher"
        use_rms=True,               # True / False
        cb_gamma=0.32,              # 0.32 / 0.0
        wa_beta=0.80,               # 0.80 / 0.0
        alpha_lin=0.45,
        seed=1993
    ):
        if skip_mode == "concat":
            head_dim = in_features + orth_dim
            m_unique = in_features + num_heads * orth_dim
        elif skip_mode in ("none", "add_1536", "add_768"):
            head_dim = orth_dim
            m_unique = num_heads * orth_dim
        else:
            raise ValueError(f"Unknown skip_mode: {skip_mode}")

        super().__init__(m_unique, num_heads, head_dim, num_classes)
        self.in_features = in_features
        self.orth_dim = orth_dim
        self.skip_mode = skip_mode
        self.use_centering = use_centering
        self.basis_mode = basis_mode
        self.proj_mode = proj_mode
        self.use_rms = use_rms
        self.class_balance_gamma = cb_gamma
        self.task_wa_beta = wa_beta
        self.alpha_spline = alpha_lin
        self.kac_gain = 1.35
        self.task_count = 0
        self.task_class_ranges = []

        self.register_buffer("global_rms", torch.tensor(0.0, dtype=torch.float32))
        self.register_buffer("G", torch.zeros(self.num_heads, self.head_dim, self.head_dim, dtype=torch.float32))
        self.register_buffer("Q", torch.zeros(self.num_heads, self.head_dim, num_classes, dtype=torch.float32))
        self.register_buffer("class_counts", torch.zeros(num_classes, dtype=torch.float64))
        self.register_buffer("weight", torch.zeros(self.num_heads, num_classes, self.head_dim, dtype=torch.float32))

        g = torch.Generator().manual_seed(seed)
        total_orth_cols = self.num_heads * self.orth_dim
        if proj_mode == "haar_qr":
            num_qr_blocks = (total_orth_cols + in_features - 1) // in_features
            orth_blocks = []
            for _ in range(num_qr_blocks):
                Z = torch.randn(in_features, in_features, generator=g)
                Q_qr, R_qr = torch.linalg.qr(Z)
                d_sign = torch.diag(R_qr).sign()
                d_sign[d_sign == 0] = 1.0
                orth_blocks.append(Q_qr * d_sign.unsqueeze(0))
            W_all = torch.cat(orth_blocks, dim=1)[:, :total_orth_cols]
        elif proj_mode == "gaussian_iid":
            W_all = torch.randn(in_features, total_orth_cols, generator=g) / math.sqrt(in_features)
        elif proj_mode == "rademacher":
            W_all = (torch.randint(0, 2, (in_features, total_orth_cols), generator=g).float() * 2.0 - 1.0) / math.sqrt(in_features)
        else:
            raise ValueError(f"Unknown proj_mode: {proj_mode}")

        self.register_buffer("Q_orth", W_all)

        k1 = torch.linspace(0.15, 0.70, self.orth_dim)
        k2 = torch.linspace(0.70, 1.45, self.orth_dim)
        k3 = torch.linspace(1.45, 2.20, self.orth_dim)
        self.register_buffer("knot_1", k1.view(1, 1, self.orth_dim))
        self.register_buffer("knot_2", k2.view(1, 1, self.orth_dim))
        self.register_buffer("knot_3", k3.view(1, 1, self.orth_dim))

    @torch.no_grad()
    def init_global_scale(self, task_f):
        if self.global_rms.item() == 0.0:
            if self.use_rms:
                rms_val = (task_f.float().norm(p=2, dim=-1) / math.sqrt(self.in_features)).mean().item()
                self.global_rms.fill_(max(rms_val, 1e-4))
            else:
                self.global_rms.fill_(1.0)

    def expand_features(self, f):
        f_32 = f.float()
        B = f_32.shape[0]
        scale = self.global_rms.clamp(min=1.0) if self.use_rms else torch.tensor(1.0, device=f.device)
        f_scaled = f_32 / scale
        z = (f_scaled @ self.Q_orth).view(B, self.num_heads, self.orth_dim)

        if self.basis_mode == "1_relu":
            s0 = F.relu(z)
            psi_raw = (scale * self.kac_gain) * s0
        elif self.basis_mode == "2_relu":
            s0 = F.relu(z)
            s1 = F.relu(z - self.knot_1)
            psi_raw = (scale * self.kac_gain) * (s0 + self.alpha_spline * (0.45 * s1))
        elif self.basis_mode == "3_relu":
            s0 = F.relu(z)
            s1 = F.relu(z - self.knot_1)
            s2 = F.relu(z - self.knot_2)
            psi_raw = (scale * self.kac_gain) * (s0 + self.alpha_spline * (0.45 * s1 + 0.25 * s2))
        elif self.basis_mode == "4_relu":
            s0 = F.relu(z)
            s1 = F.relu(z - self.knot_1)
            s2 = F.relu(z - self.knot_2)
            s3 = F.relu(z - self.knot_3)
            psi_raw = (scale * self.kac_gain) * (s0 + self.alpha_spline * (0.45 * s1 + 0.25 * s2 + 0.15 * s3))
        elif self.basis_mode == "4_relu_tanh":
            s0 = F.relu(z)
            s1 = F.relu(z - self.knot_1)
            s2 = F.relu(z - self.knot_2)
            s3 = F.relu(z - self.knot_3)
            s_int_rbf = torch.tanh(s0)
            psi_raw = (scale * self.kac_gain) * (
                s0 + self.alpha_spline * (0.45 * s1 + 0.25 * s2 + 0.15 * s3 + 0.35 * s_int_rbf)
            )
        elif self.basis_mode == "full_v20":
            s0 = F.relu(z)
            s1 = F.relu(z - self.knot_1)
            s2 = F.relu(z - self.knot_2)
            s3 = F.relu(z - self.knot_3)
            s_int_rbf = torch.tanh(s0)
            s0_adj1 = torch.roll(s0, shifts=1, dims=-1)
            s0_adj2 = torch.roll(s0, shifts=2, dims=-1)
            s_had1 = (s0 * s0_adj1) / (1.0 + 0.5 * (s0 + s0_adj1))
            s_had2 = (s0 * s0_adj2) / (1.0 + 0.5 * (s0 + s0_adj2))
            psi_raw = (scale * self.kac_gain) * (
                s0 + self.alpha_spline * (
                    0.45 * s1 + 0.25 * s2 + 0.15 * s3 + 0.35 * s_int_rbf + 0.15 * s_had1 + 0.10 * s_had2
                )
            )
        elif self.basis_mode == "gaussian_rbf":
            g1 = torch.exp(-0.5 * ((z + 1.5) ** 2))
            g2 = torch.exp(-0.5 * ((z + 0.5) ** 2))
            g3 = torch.exp(-0.5 * ((z - 0.5) ** 2))
            g4 = torch.exp(-0.5 * ((z - 1.5) ** 2))
            psi_raw = (scale * self.kac_gain) * (0.25 * g1 + 0.35 * g2 + 0.45 * g3 + 0.35 * g4)
        else:
            raise ValueError(f"Unknown basis_mode: {self.basis_mode}")

        psi_orth = (psi_raw - psi_raw.mean(dim=-1, keepdim=True)) if self.use_centering else psi_raw

        if self.skip_mode == "concat":
            f_expand = f_32.unsqueeze(1).expand(B, self.num_heads, self.in_features)
            phi = torch.cat([f_expand, psi_orth], dim=-1)
        elif self.skip_mode == "none":
            phi = psi_orth
        elif self.skip_mode == "add_768":
            f_expand = f_32.unsqueeze(1).expand(B, self.num_heads, self.in_features)
            phi = f_expand + psi_orth
        elif self.skip_mode == "add_1536":
            f_tiled = torch.cat([f_32, f_32], dim=-1).unsqueeze(1).expand(B, self.num_heads, self.orth_dim) / math.sqrt(2.0)
            phi = f_tiled + psi_orth
        return phi

    @torch.no_grad()
    def update_and_solve(self, task_f, task_y, candidate_lambdas=None, task_range=None):
        if candidate_lambdas is None:
            candidate_lambdas = [10.0**p for p in range(-8, 9)]
        self.init_global_scale(task_f)

        if task_range is not None:
            t_lo, t_hi = task_range
        else:
            t_lo, t_hi = int(task_y.min().item()), int(task_y.max().item()) + 1
        if (t_lo, t_hi) not in self.task_class_ranges:
            self.task_class_ranges.append((t_lo, t_hi))

        task_h = self.expand_features(task_f)
        N = task_h.shape[0]
        task_h_32 = task_h.permute(1, 0, 2)
        task_Y_32 = F.one_hot(task_y, num_classes=self.num_classes).to(torch.float32)

        self.task_count += 1
        is_imbal = False
        if len(self.task_class_ranges) > 0:
            seen = self.class_counts > 0
            if seen.any() and (self.class_counts[seen].std() / (self.class_counts[seen].mean() + 1e-8)).item() > 0.08:
                is_imbal = True
        min_lam = 3000.0 if is_imbal else 100.0

        raw_lambda = self.find_optimal_lambda_fast(task_h[:, 0, :], task_y, candidate_lambdas, min_lambda=min_lam)
        if self.task_count == 1:
            self.base_lambda = raw_lambda
        else:
            self.base_lambda = (self.base_lambda ** 0.85) * (raw_lambda ** 0.15)
        self.optimal_lambda = self.base_lambda * (self.task_count ** 0.15)

        self.G += torch.bmm(task_h_32.transpose(1, 2), task_h_32)
        self.Q += torch.matmul(task_h_32.transpose(1, 2), task_Y_32.unsqueeze(0))
        self.class_counts += task_Y_32.to(torch.float64).sum(dim=0)

        seen_mask = self.class_counts > 0
        cls_scale = torch.ones_like(self.class_counts)
        is_imbalanced_dataset = False
        if seen_mask.any():
            counts_seen = self.class_counts[seen_mask]
            cv_counts = (counts_seen.std() / (counts_seen.mean() + 1e-8)).item()
            if cv_counts > 0.08:
                is_imbalanced_dataset = True
                if self.class_balance_gamma > 0.0:
                    mean_count = counts_seen.mean()
                    cls_scale[seen_mask] = (mean_count / counts_seen).pow(self.class_balance_gamma)

        I_mat = torch.eye(self.head_dim, device=self.G.device, dtype=torch.float64)
        W_list = []
        for h in range(self.num_heads):
            G_reg = self.G[h].to(torch.float64) + self.optimal_lambda * I_mat
            Q_h = self.Q[h].to(torch.float64) * cls_scale.unsqueeze(0)
            L_h, info = torch.linalg.cholesky_ex(G_reg)
            if info.item() == 0:
                W_h = torch.cholesky_solve(Q_h, L_h)
            else:
                W_h = torch.linalg.solve(G_reg, Q_h)
            W_list.append(W_h.float())
        W_opt = torch.stack(W_list, dim=0)
        W_t = W_opt.transpose(1, 2)

        eff_beta = self.task_wa_beta if is_imbalanced_dataset else (0.20 if self.task_wa_beta > 0 else 0.0)
        if len(self.task_class_ranges) > 1 and eff_beta > 0.0:
            task_norms = []
            for (lo, hi) in self.task_class_ranges:
                w_task = W_t[:, lo:hi, :]
                rms_task = w_task.pow(2).sum(dim=-1, keepdim=True).mean(dim=1, keepdim=True).sqrt()
                task_norms.append(rms_task)
            mean_task_norm = torch.stack(task_norms, dim=0).mean(dim=0)
            for (lo, hi), r_t in zip(self.task_class_ranges, task_norms):
                scale_t = (mean_task_norm / (r_t + 1e-8)).pow(eff_beta)
                W_t[:, lo:hi, :] *= scale_t

        self.weight.copy_(W_t)
        self.total_flops_accum += (2.0 * N * self.in_features * (self.num_heads * self.orth_dim) +
                                   self.num_heads * (2.0 * N * (self.head_dim**2) + 2.0 * N * self.head_dim * self.num_classes) +
                                   self.num_heads * ((1.0 / 3.0) * (self.head_dim**3) + 2.0 * (self.head_dim**2) * self.num_classes))

    def forward(self, f):
        h = self.expand_features(f)
        return torch.einsum("bhd,hcd->bc", h, self.weight) / self.num_heads

def build_model_head(model_id, num_classes=200, seed=1993):
    spec = ABLATION_REGISTRY[model_id]
    arch = spec["arch"]
    if arch == "linear":
        return LinearRidgeHead(in_features=768, num_classes=num_classes).to(DEVICE)
    elif arch == "kac_axis":
        return AxisKACRidgeHead(
            in_features=768, num_classes=num_classes,
            num_grids=spec.get("num_grids", 4), grid_range=spec.get("grid_range", (-2.0, 2.0))
        ).to(DEVICE)
    elif arch == "ranpac":
        return RanPACHead(in_features=768, num_classes=num_classes, M=spec["M"], seed=seed).to(DEVICE)
    elif arch == "so_kac":
        return ConfigurableSOKACHead(
            in_features=768, num_classes=num_classes,
            num_heads=spec["num_heads"], orth_dim=spec["orth_dim"],
            skip_mode=spec["skip_mode"], use_centering=spec["use_centering"],
            basis_mode=spec["basis_mode"], proj_mode=spec["proj_mode"],
            use_rms=spec["use_rms"], cb_gamma=spec["cb_gamma"], wa_beta=spec["wa_beta"],
            seed=seed
        ).to(DEVICE)
    else:
        raise ValueError(f"Unknown arch: {arch}")

print("✅ Đã định nghĩa đầy đủ các Bộ Phân Loại Nghiệm Đóng & ConfigurableSOKACHead.")


In [ ]:
# ==============================================================================
# CELL 7: ENGINE ĐÁNH GIÁ CHẨN ĐOÁN & THỰC THI 2 GIAI ĐOẠN (VAL ABLATION / FINAL TEST)
# ==============================================================================
@torch.no_grad()
def compute_feature_eff_rank(f_mat):
    f_c = f_mat - f_mat.mean(dim=0, keepdim=True)
    cov = (f_c.T @ f_c) / max(len(f_mat) - 1, 1)
    eigvals = torch.linalg.eigvalsh(cov.to(torch.float64)).clamp(min=0.0)
    S = eigvals.sqrt()
    S = S / (S.sum() + 1e-12)
    entropy = -(S * torch.log(S + 1e-12)).sum()
    return float(torch.exp(entropy).item())

@torch.no_grad()
def compute_gram_eff_rank(G_head0):
    eigvals = torch.linalg.eigvalsh(G_head0.to(torch.float64)).clamp(min=0.0)
    S = eigvals.sqrt()
    S = S / (S.sum() + 1e-12)
    entropy = -(S * torch.log(S + 1e-12)).sum()
    return float(torch.exp(entropy).item())

@torch.no_grad()
def evaluate_head(head, cached_ev_f, cached_ev_y, n_seen_tasks, bounds, eff_rank_vit=0.0, compute_phi_rank=False):
    hi = bounds[n_seen_tasks - 1][1]
    mask_seen = (cached_ev_y < hi)
    f_all = cached_ev_f[mask_seen]
    y = cached_ev_y[mask_seen]

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t_inf0 = time.perf_counter()

    B_eval = 512
    logits_list = []
    for i in range(0, len(f_all), B_eval):
        logits_list.append(head(f_all[i:i + B_eval])[:, :hi])
    logits = torch.cat(logits_list, dim=0)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    infer_time_ms = (time.perf_counter() - t_inf0) * 1000.0

    if hasattr(head, "Q_orth"):
        in_dim = min(getattr(head, "in_features", 768), head.Q_orth.shape[1])
        scale = head.global_rms.clamp(min=1.0) if getattr(head, "use_rms", True) else 1.0
        z_check = (f_all / scale) @ head.Q_orth[:, :in_dim]
        u_out = ((z_check < -2.0) | (z_check > 2.0)).float().mean().item() * 100.0
    elif isinstance(head, AxisKACRidgeHead):
        u_norm = F.layer_norm(f_all.float(), (head.in_features,))
        u_out = ((u_norm < -2.0) | (u_norm > 2.0)).float().mean().item() * 100.0
    else:
        u_out = 0.0

    correct_ta = 0
    accs = []
    preds = logits.argmax(dim=1)
    for j in range(n_seen_tasks):
        t_lo, t_hi = bounds[j]
        mask = (y >= t_lo) & (y < t_hi)
        if mask.sum() > 0:
            pred_ta = logits[mask, t_lo:t_hi].argmax(dim=1) + t_lo
            correct_ta += (pred_ta == y[mask]).sum().item()
            accs.append(100.0 * (preds[mask] == y[mask]).float().mean().item())
        else:
            accs.append(0.0)
    acc_ta = 100.0 * correct_ta / max(len(y), 1)
    overall = 100.0 * (preds == y).float().mean().item()
    eff_rank_phi = compute_gram_eff_rank(head.G[0]) if compute_phi_rank else 0.0

    diag = {
        "eff_rank": round(eff_rank_vit, 2),
        "eff_rank_phi": round(eff_rank_phi, 2),
        "u_out_of_grid": round(u_out, 2),
        "acc_ta": round(acc_ta, 2),
        "optimal_lambda": float(head.optimal_lambda),
        "infer_time_ms": round(infer_time_ms, 2),
    }
    return accs, overall, diag

def get_hardware_stats(head):
    gram_ram_mb = (head.G.numel() * head.G.element_size()) / (1024**2)
    total_buf_mb = sum(b.numel() * b.element_size() for b in head.buffers()) / (1024**2)
    cholesky_macs_per_task = head.num_heads * ((1.0 / 6.0) * (head.head_dim ** 3))
    return dict(
        M_unique=int(head.M), num_heads=int(head.num_heads), head_dim=int(head.head_dim),
        config_str=f"{head.num_heads}x{head.head_dim}",
        gram_ram_mb=round(gram_ram_mb, 2), total_buf_mb=round(total_buf_mb, 2),
        cholesky_gmacs=round(cholesky_macs_per_task / 1e9, 3),
        total_gflops_10t=round(head.total_flops_accum / 1e9, 2),
    )

def summarize(R):
    Tn = R.shape[0]
    A = [float(np.mean(R[t, :t + 1])) for t in range(Tn)]
    fgt = float(np.mean([R[j, j] - R[Tn - 1, j] for j in range(Tn - 1)])) if Tn > 1 else 0.0
    return dict(A_t=A, last=A[-1], avg=float(np.mean(A)), forgetting=fgt)

def run_dataset_phase_seed(d_key, seed, phase, model_ids):
    """
    Thực thi 1 Dataset x 1 Seed ở chế độ `phase`:
      - phase == "val"  : Huấn luyện trên 80% Train (tr_idx), đánh giá trên 20% Val (va_idx). TUYỆT ĐỐI KHÔNG ĐỤNG TEST.
      - phase == "test" : Huấn luyện lại trên 100% Train gốc, đánh giá trên tập Test chính thức.
    """
    tr_ds, te_ds, _ = AVAILABLE_DATASETS[d_key]
    spec = DATASET_SPECS[d_key]
    C = spec["n_classes"]
    task_sizes = spec["task_sizes"]
    nT = len(task_sizes)
    Tn = nT if MAX_TASKS_DEBUG is None else min(nT, MAX_TASKS_DEBUG)
    E1, lr, ema, bs = spec["first_session_epochs"], spec["prompt_lr"], spec["ema_coeff"], spec["batch_size"]
    train_tf, test_tf = build_transforms(spec["aug_type"])

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    _, remap, bounds, tr_idx, va_idx = setup_cil_seed(tr_ds, seed, C, task_sizes)
    tr_targets_all = np.asarray(tr_ds.targets)

    if phase == "val":
        active_train_indices = tr_idx
        ev_dataset = RemappedSubset(tr_ds, va_idx, remap, test_tf)
        phase_label = "PHASE 1: VAL ABLATION (Train 80% -> Val 20% Disjoint, Test Untouched)"
    elif phase == "test":
        active_train_indices = np.arange(len(tr_ds))
        ev_dataset = RemappedSubset(te_ds, np.arange(len(te_ds)), remap, test_tf)
        phase_label = "PHASE 2: FINAL TEST (Train 100% Full -> Official Test)"
    else:
        raise ValueError("phase phải là 'val' hoặc 'test'")

    print(f"\n{'='*100}")
    print(f"  🚀 [{spec['dataset_name']}] {phase_label} | SEED {seed} | Ep={E1}, LR={lr}, EMA={ema}")
    print(f"{'='*100}")

    # 1. Huấn luyện APT Prompt ở Task 1
    t1_lo, t1_hi = bounds[0]
    t1_train_idx = [i for i in active_train_indices if t1_lo <= remap[int(tr_targets_all[i])] < t1_hi]
    t1_ds = RemappedSubset(tr_ds, t1_train_idx, remap, train_tf)
    prompt, avg_p_grad, avg_c_grad = train_first_session_prompt(t1_ds, t1_hi - t1_lo, E1, lr, ema, bs)

    # 2. Trích xuất & Cache đặc trưng 1 lần duy nhất cho toàn bộ các mô hình
    t_ext0 = time.perf_counter()
    ev_loader = DataLoader(ev_dataset, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    cached_ev_f, cached_ev_y = [], []
    with torch.no_grad():
        for x, y in ev_loader:
            x = x.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE == "cuda"):
                f = VIT(x, prompt_fn=prompt.get)
            cached_ev_f.append(f.float())
            cached_ev_y.append(y.to(DEVICE))
    cached_ev_f = torch.cat(cached_ev_f, dim=0)
    cached_ev_y = torch.cat(cached_ev_y, dim=0)

    cached_train_tasks, eff_ranks_vit_per_task = [], []
    for t in range(Tn):
        lo, hi = bounds[t]
        t_idx = [i for i in active_train_indices if lo <= remap[int(tr_targets_all[i])] < hi]
        t_loader = DataLoader(RemappedSubset(tr_ds, t_idx, remap, test_tf), batch_size=bs,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
        t_f, t_y = [], []
        with torch.no_grad():
            for x, y in t_loader:
                x = x.to(DEVICE, non_blocking=True)
                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP and DEVICE == "cuda"):
                    f = VIT(x, prompt_fn=prompt.get)
                t_f.append(f.float())
                t_y.append(y.to(DEVICE))
        cached_train_tasks.append((torch.cat(t_f, dim=0), torch.cat(t_y, dim=0), lo, hi))
        eff_ranks_vit_per_task.append(compute_feature_eff_rank(cached_ev_f[cached_ev_y < hi]))

    print(f"    [*] Đã trích xuất đặc trưng ({len(active_train_indices)} Train / {len(ev_dataset)} Eval) trong {time.perf_counter()-t_ext0:.1f}s.")
    print("-" * 100)

    # 3. Đánh giá tuần tự từng cấu hình trên đặc trưng đã cache
    seed_results = OrderedDict()
    for idx_m, mid in enumerate(model_ids, 1):
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            base_vram = torch.cuda.memory_allocated()
        else:
            base_vram = 0

        head = build_model_head(mid, num_classes=C, seed=1993)
        R_mat = np.zeros((Tn, Tn))
        task_logs, solve_times = [], []

        for t in range(Tn):
            t_f, t_y, lo, hi = cached_train_tasks[t]
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            head.update_and_solve(t_f, t_y, RIDGE_LAMBDAS, task_range=(lo, hi))
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t_solve = time.perf_counter() - t0
            solve_times.append(t_solve)

            accs, overall, diag = evaluate_head(
                head, cached_ev_f, cached_ev_y, t + 1, bounds,
                eff_rank_vit=eff_ranks_vit_per_task[t], compute_phi_rank=(t == Tn - 1)
            )
            diag["p_grad"] = avg_p_grad
            diag["c_grad"] = avg_c_grad
            R_mat[t, :t + 1] = accs
            task_logs.append(dict(task=t + 1, A_t=float(np.mean(accs)), sample_acc=overall, solve_time=round(t_solve, 4), accs=accs, diag=diag))

        peak_vram_mb = max((torch.cuda.max_memory_allocated() - base_vram) / (1024**2), 0.0) if DEVICE == "cuda" else 0.0
        hw = get_hardware_stats(head)
        hw.update(
            peak_vram_mb=round(peak_vram_mb, 2),
            mean_solve_s=round(float(np.mean(solve_times)), 3),
            total_solve_s=round(float(np.sum(solve_times)), 2),
            infer_time_ms=task_logs[-1]["diag"]["infer_time_ms"],
        )

        res = summarize(R_mat)
        last_diag = task_logs[-1]["diag"]
        res.update(
            dataset=spec["dataset_name"], phase=phase, ablation_id=mid, desc=ABLATION_REGISTRY[mid]["desc"], seed=seed,
            acc_ta=last_diag["acc_ta"], eff_rank=last_diag["eff_rank"], eff_rank_phi=last_diag["eff_rank_phi"],
            u_out_of_grid=last_diag["u_out_of_grid"], optimal_lambda=last_diag["optimal_lambda"],
            p_grad=avg_p_grad, c_grad=avg_c_grad, hw=hw, R=R_mat.tolist(), log=task_logs,
        )
        seed_results[mid] = res
        print(f"  [{idx_m:2d}/{len(model_ids):2d}] {mid:<22} | Last: {res['last']:6.2f}% | Avg: {res['avg']:6.2f}% | "
              f"Fgt: {res['forgetting']:5.2f}% | TA: {res['acc_ta']:6.2f}% | RAM: {hw['gram_ram_mb']:6.1f}MB | Solve: {hw['mean_solve_s']:.2f}s/T")

        del head
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    del prompt, cached_ev_f, cached_ev_y, cached_train_tasks
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return seed_results

def aggregate_seed_runs(d_name, phase, model_ids, runs_by_model):
    summary = OrderedDict()
    for mid in model_ids:
        runs = runs_by_model[mid]
        if not runs:
            continue
        hw0 = runs[0]["hw"]
        summary[mid] = dict(
            dataset=d_name, phase=phase, ablation_id=mid, desc=ABLATION_REGISTRY[mid]["desc"],
            seeds=len(runs), config_str=hw0["config_str"], M_unique=hw0["M_unique"],
            last_mean=float(np.mean([r["last"] for r in runs])), last_std=float(np.std([r["last"] for r in runs])),
            avg_mean=float(np.mean([r["avg"] for r in runs])), avg_std=float(np.std([r["avg"] for r in runs])),
            fgt_mean=float(np.mean([r["forgetting"] for r in runs])), fgt_std=float(np.std([r["forgetting"] for r in runs])),
            ta_mean=float(np.mean([r["acc_ta"] for r in runs])), ta_std=float(np.std([r["acc_ta"] for r in runs])),
            eff_rank_vit=runs[0]["eff_rank"],
            eff_rank_phi=float(np.mean([r["eff_rank_phi"] for r in runs])),
            u_out_of_grid=float(np.mean([r["u_out_of_grid"] for r in runs])),
            gram_ram_mb=hw0["gram_ram_mb"],
            peak_vram_mb=float(np.mean([r["hw"]["peak_vram_mb"] for r in runs])),
            cholesky_gmacs=hw0["cholesky_gmacs"],
            total_gflops_10t=hw0["total_gflops_10t"],
            solve_time_s=float(np.mean([r["hw"]["mean_solve_s"] for r in runs])),
            infer_time_ms=float(np.mean([r["hw"]["infer_time_ms"] for r in runs])),
        )
    return summary


In [ ]:
# ==============================================================================
# CELL 8: CHẠY THỰC NGHIỆM (VAL ABLATION + FINAL TEST), IN BẢO CÁO & NÉN ZIP 1-CLICK
# ==============================================================================
def format_markdown_table(summary_dict, model_id_list):
    lines = [
        f"| {'Mã Cấu Hình':<22} | {'Cỡ Gram':<11} | {'RAM G(MB)':>9} | {'PeakVRAM':>8} | "
        f"{'Last Acc (%)':<15} | {'Avg Acc (%)':<15} | {'Fgt (%)':<12} | {'TA Acc (%)':<10} | "
        f"{'Solve(s/T)':>10} | {'Mô Tả Kiến Trúc'} |",
        f"| :{'-'*20} | :{'-'*9}: | {'-'*8}: | {'-'*7}: | "
        f":{'-'*13}: | :{'-'*13}: | :{'-'*10}: | :{'-'*8}: | "
        f"{'-'*9}: | :{'-'*25} |",
    ]
    for mid in model_id_list:
        if mid not in summary_dict:
            continue
        s = summary_dict[mid]
        last_str = f"{s['last_mean']:.2f} ± {s['last_std']:.2f}" if s["seeds"] > 1 else f"{s['last_mean']:.2f}"
        avg_str  = f"{s['avg_mean']:.2f} ± {s['avg_std']:.2f}"  if s["seeds"] > 1 else f"{s['avg_mean']:.2f}"
        fgt_str  = f"{s['fgt_mean']:.2f} ± {s['fgt_std']:.2f}"  if s["seeds"] > 1 else f"{s['fgt_mean']:.2f}"
        lines.append(
            f"| **{mid:<18}** | `{s['config_str']:<9}` | {s['gram_ram_mb']:>7.1f}MB | {s['peak_vram_mb']:>6.1f}MB | "
            f"**{last_str:<11}** | {avg_str:<15} | {fgt_str:<12} | {s['ta_mean']:>8.2f}% | "
            f"{s['solve_time_s']:>8.2f}s | {s['desc']} |"
        )
    return "\n".join(lines)

def export_csv_results(d_out_dir, d_name, phase, runs_by_model, summary_dict):
    task_rows = []
    for mid, runs in runs_by_model.items():
        for r in runs:
            s = r["seed"]
            for step in r.get("log", []):
                diag = step.get("diag", {})
                task_rows.append({
                    "dataset": d_name, "phase": phase, "seed": s, "model_id": mid,
                    "desc": r.get("desc", ""), "task": step["task"], "A_t": step["A_t"],
                    "sample_acc": step["sample_acc"], "acc_ta": diag.get("acc_ta", 0.0),
                    "eff_rank_vit": diag.get("eff_rank", 0.0), "eff_rank_phi": diag.get("eff_rank_phi", 0.0),
                    "u_out_of_grid": diag.get("u_out_of_grid", 0.0), "optimal_lambda": diag.get("optimal_lambda", 0.0),
                    "solve_time_s": step.get("solve_time", 0.0), "infer_time_ms": diag.get("infer_time_ms", 0.0),
                })
    if task_rows:
        p_task_csv = os.path.join(d_out_dir, f"APT_{d_name}_{phase.upper()}_task_progression.csv")
        pd.DataFrame(task_rows).to_csv(p_task_csv, index=False)
        print(f"  📊 [CSV] Đã xuất chi tiết từng Task: {p_task_csv}")

    sum_rows = []
    for mid, s in summary_dict.items():
        sum_rows.append({
            "dataset": d_name, "phase": phase, "model_id": mid,
            "desc": s.get("desc", ""), "config_str": s.get("config_str", ""), "seeds": s.get("seeds", 1),
            "last_mean": s.get("last_mean", 0.0), "last_std": s.get("last_std", 0.0),
            "avg_mean": s.get("avg_mean", 0.0), "avg_std": s.get("avg_std", 0.0),
            "fgt_mean": s.get("fgt_mean", 0.0), "fgt_std": s.get("fgt_std", 0.0),
            "ta_mean": s.get("ta_mean", 0.0), "ta_std": s.get("ta_std", 0.0),
            "eff_rank": s.get("eff_rank", 0.0), "eff_rank_phi": s.get("eff_rank_phi", 0.0),
            "gram_ram_mb": s.get("gram_ram_mb", 0.0), "peak_vram_mb": s.get("peak_vram_mb", 0.0),
            "mean_solve_s": s.get("mean_solve_s", 0.0), "total_solve_s": s.get("total_solve_s", 0.0),
            "infer_time_ms": s.get("infer_time_ms", 0.0),
        })
    if sum_rows:
        p_sum_csv = os.path.join(d_out_dir, f"APT_{d_name}_{phase.upper()}_model_summary.csv")
        pd.DataFrame(sum_rows).to_csv(p_sum_csv, index=False)
        print(f"  📊 [CSV] Đã xuất tổng hợp Model Summary: {p_sum_csv}")

ALL_REPORTS = []

for d_key in AVAILABLE_DATASETS:
    spec = DATASET_SPECS[d_key]
    d_name = spec["dataset_name"]
    d_out_dir = os.path.join(MASTER_OUT_DIR, d_name)
    os.makedirs(d_out_dir, exist_ok=True)
    ALL_REPORTS.append(f"# 📊 BÁO CÁO DATASET: **{d_name}** (LR={spec['prompt_lr']}, Epochs={spec['first_session_epochs']}, EMA={spec['ema_coeff']})")

    # --- GIAI ĐOẠN 1: CHẠY ĐỦ 7 NHÓM / 31 CẤU HÌNH ABLATION TRÊN TẬP VAL 80/20 ---
    if RUN_VAL_ABLATION:
        val_runs = OrderedDict((mid, []) for mid in ALL_ABLATION_IDS)
        for seed in SEEDS_TO_RUN:
            s_res = run_dataset_phase_seed(d_key, seed, phase="val", model_ids=ALL_ABLATION_IDS)
            for mid, r in s_res.items():
                val_runs[mid].append(r)
            p_seed = os.path.join(d_out_dir, f"APT_{d_name}_VAL_Ablation_seed{seed}.json")
            with open(p_seed, "w", encoding="utf-8") as f:
                json.dump(list(s_res.values()), f, indent=2, ensure_ascii=False)

        val_summary = aggregate_seed_runs(d_name, "val", ALL_ABLATION_IDS, val_runs)
        p_val_sum = os.path.join(d_out_dir, f"APT_{d_name}_VAL_Ablation_Summary.json")
        with open(p_val_sum, "w", encoding="utf-8") as f:
            json.dump(list(val_summary.values()), f, indent=2, ensure_ascii=False)
        export_csv_results(d_out_dir, d_name, "val", val_runs, val_summary)

        print(f"\n{'='*100}\n📊 BẢNG TỔNG HỢP TRUNG BÌNH {len(SEEDS_TO_RUN)} SEEDS (MEAN ± STD) — GIAI ĐOẠN 1 (VAL ABLATION)\n{'='*100}")
        ALL_REPORTS.append(f"\n## 🔹 GIAI ĐOẠN 1: KẾT QUẢ 7 NHÓM ABLATION TRÊN TẬP VAL (80/20 DISJOINT, SEEDS={SEEDS_TO_RUN})")
        for grp_key, (grp_title, grp_mids) in ABLATION_GROUPS.items():
            tbl_md = format_markdown_table(val_summary, grp_mids)
            ALL_REPORTS.append(f"\n### {grp_title}")
            ALL_REPORTS.append(tbl_md)
            print(f"\n### {grp_title}\n{tbl_md}")

        ranked_val_mids = sorted(val_summary.keys(), key=lambda m: val_summary[m]["last_mean"], reverse=True)
        print(f"\n🏆 BẢNG XẾP HẠNG TOP CẤU HÌNH TỐI ƯU TRÊN VAL (SORTED BY LAST ACC):")
        for rank, m in enumerate(ranked_val_mids[:10], 1):
            s = val_summary[m]
            print(f"  {rank:2d}. {m:<22} | Last: {s['last_mean']:5.2f} ± {s['last_std']:4.2f}% | Avg: {s['avg_mean']:5.2f}% | Fgt: {s['fgt_mean']:4.2f}% | TA: {s['ta_mean']:5.2f}% | RAM: {s['gram_ram_mb']:5.1f}MB | Solve: {s['solve_time_s']:4.2f}s")

    # --- GIAI ĐOẠN 2: CHẠY FINAL TEST TRÊN 100% TRAIN GỐC + TẬP TEST CHÍNH THỨC ---
    if RUN_FINAL_TEST:
        test_runs = OrderedDict((mid, []) for mid in FINAL_TEST_MODELS)
        for seed in SEEDS_TO_RUN:
            s_res = run_dataset_phase_seed(d_key, seed, phase="test", model_ids=FINAL_TEST_MODELS)
            for mid, r in s_res.items():
                test_runs[mid].append(r)
            p_seed = os.path.join(d_out_dir, f"APT_{d_name}_FINAL_TEST_seed{seed}.json")
            with open(p_seed, "w", encoding="utf-8") as f:
                json.dump(list(s_res.values()), f, indent=2, ensure_ascii=False)

        test_summary = aggregate_seed_runs(d_name, "test", FINAL_TEST_MODELS, test_runs)
        p_test_sum = os.path.join(d_out_dir, f"APT_{d_name}_FINAL_TEST_Summary.json")
        with open(p_test_sum, "w", encoding="utf-8") as f:
            json.dump(list(test_summary.values()), f, indent=2, ensure_ascii=False)
        export_csv_results(d_out_dir, d_name, "test", test_runs, test_summary)

        test_tbl_md = format_markdown_table(test_summary, FINAL_TEST_MODELS)
        ALL_REPORTS.append(f"\n## 🏆 GIAI ĐOẠN 2: KẾT QUẢ FINAL BENCHMARK TRÊN TẬP TEST CHÍNH THỨC (100% TRAIN, SEEDS={SEEDS_TO_RUN})")
        ALL_REPORTS.append(test_tbl_md)
        print(f"\n{'='*100}\n🏆 BẢNG TỔNG HỢP TRUNG BÌNH {len(SEEDS_TO_RUN)} SEEDS (MEAN ± STD) — GIAI ĐOẠN 2 (FINAL TEST CHÍNH THỨC)\n{'='*100}")
        print(test_tbl_md)

report_md = "\n".join(ALL_REPORTS) + "\n"
print("\n" + report_md)

report_path = os.path.join(MASTER_OUT_DIR, "APT_SO_KAC_Full_Ablation_And_FinalTest_Report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

zip_base = os.path.join(os.path.dirname(MASTER_OUT_DIR), "APT_SO_KAC_Full_Ablation_And_FinalTest_Results")
shutil.make_archive(zip_base, "zip", MASTER_OUT_DIR)
print(f"\n✅ Đã lưu toàn bộ JSON, Markdown Report và nén file Zip tại: {zip_base}.zip")
